# gsd — quickstart for practitioners

The minimum you need to **apply** the library to your own series. We fit the
same data two ways:

- **OLS** — ordinary least squares on lags: `AR(p)` (one series) / `VAR(p)`
  (several). The familiar regression table.
- **SPTLS** — the one-step fit on the kinematic embedding
  $q(t)=(z,\Delta z,\Delta^2 z)$, the programme's estimator.

Here we cover fitting, the summary table, one-step forecasting, and
walk-forward (out-of-sample) validation — univariate and multivariate. The
**interpretation/diagnostic suite** (rotor, spectrum, grade energy,
cross-effects, shared directions) is a separate read; see the companion
notebook `gsd_diagnostics.ipynb`.

## Setup

`gsd` needs only **numpy** (and **matplotlib** for plots). Point Python at the
library folder, then import.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../lib"))   # repo-relative path to lib/gsd
import numpy as np
import gsd
np.set_printoptions(precision=3, suppress=True)
print("gsd", gsd.__version__)

## 1. A univariate series

We use a synthetic damped oscillation shipped with the library; swap in your
own 1-D array anywhere below.

In [ ]:
y = gsd.datasets.rotation(T=200)
print("series length:", len(y))

## 2. OLS — the regression baseline

`gsd.OLS(p).fit(y)` returns a result with a `statsmodels`-style `.summary()`
(coefficients, standard errors, t, p-values, confidence intervals, plus
R², AIC, BIC, Durbin–Watson).

In [ ]:
ols = gsd.OLS(2).fit(y)          # AR(2)
print(ols.summary())

## 3. SPTLS — the one-step embedding fit

`gsd.SPTLS().fit(y)` fits the operator on the kinematic jet. `rsquared` is the
one-step fit on the level (`z`) channel; `forecast_next` gives the next value
from a history.

In [ ]:
sp = gsd.SPTLS().fit(y)
print("SPTLS one-step R2:", round(sp.rsquared_mean, 4))
print("OLS  AR(2)    R2:", round(ols.rsquared, 4))
print("next-step forecast (SPTLS):", round(sp.forecast_next(y), 4))

## 4. Out-of-sample validation (walk-forward)

Never train on the future. `gsd.validation.compare` runs rolling-origin
one-step CV for several estimators and ranks them by RMSE.

In [ ]:
cv = gsd.validation.compare(
    {"OLS AR(2)": gsd.OLS(2), "SPTLS": gsd.SPTLS()},
    y, start=120, scheme="expanding")
print(cv["_table"])

## 5. Multivariate: VAR vs SPTLS

For several series, `gsd.OLS(p).fit(Y)` is a `VAR(p)` (one equation per
series, each with its own `.summary()`); `gsd.SPTLS().fit(Y)` stacks the jets
into one operator. Below, two cointegrated (individually non-stationary)
series.

In [ ]:
Y = gsd.datasets.cointegrated_var(T=400)
var = gsd.OLS(1).fit(Y)
print("VAR(1) per-equation R2:", var.rsquared)
spm = gsd.SPTLS().fit(Y)
print("SPTLS per-series one-step R2:", {k: round(v,4) for k,v in spm.rsquared.items()})

cvm = gsd.validation.compare(
    {"VAR(1)": gsd.OLS(1), "SPTLS": gsd.SPTLS()},
    Y, start=240, scheme="expanding")
print(cvm["_table"])

## Where next

- To **understand** a fitted process (is it rotating? damping? which series
  drives which? are there shared directions among the levels?), open the
  companion **`gsd_diagnostics.ipynb`** — the SPTLS reading suite.
- Everything here is plain numpy; drop in your own arrays and the same calls
  apply.